In [15]:
import sqlite3
import pandas as pd

# Load all datasets
customers = pd.read_csv('/Users/gurnoor21/Documents/docs/ecommerce customer behaviour analysis/archive/olist_customers_dataset.csv')
orders = pd.read_csv('/Users/gurnoor21/Documents/docs/ecommerce customer behaviour analysis/archive/olist_orders_dataset.csv')
order_items = pd.read_csv('/Users/gurnoor21/Documents/docs/ecommerce customer behaviour analysis/archive/olist_order_items_dataset.csv')
payments = pd.read_csv('/Users/gurnoor21/Documents/docs/ecommerce customer behaviour analysis/archive/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/Users/gurnoor21/Documents/docs/ecommerce customer behaviour analysis/archive/olist_order_reviews_dataset.csv')
products = pd.read_csv('/Users/gurnoor21/Documents/docs/ecommerce customer behaviour analysis/archive/olist_products_dataset.csv')

print("✅ All datasets loaded!")
print(f"Customers: {len(customers)}, Orders: {len(orders)}, Items: {len(order_items)}")

✅ All datasets loaded!
Customers: 99441, Orders: 99441, Items: 112650


In [16]:
# Create SQLite database
conn = sqlite3.connect('ecommerce_analysis.db')

# Clean and prepare data
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

# Create tables in database
customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
payments.to_sql('payments', conn, if_exists='replace', index=False)
reviews.to_sql('reviews', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

print("✅ Database created with 6 tables!")
print("Tables:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))

✅ Database created with 6 tables!
Tables:           name
0    customers
1       orders
2  order_items
3     payments
4      reviews
5     products


In [17]:
# 🔥 IMPRESSIVE QUERY #1: Customer Retention Analysis with Window Functions
retention_query = """
WITH customer_orders AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp) as order_number,
        LAG(o.order_purchase_timestamp) OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp) as previous_order_date
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
),
customer_behavior AS (
    SELECT 
        customer_unique_id,
        COUNT(*) as total_orders,
        MIN(order_purchase_timestamp) as first_order_date,
        MAX(order_purchase_timestamp) as last_order_date,
        CASE 
            WHEN COUNT(*) = 1 THEN 'One-time'
            WHEN COUNT(*) = 2 THEN 'Two-time' 
            WHEN COUNT(*) >= 3 THEN 'Loyal'
        END as customer_segment,
        JULIANDAY(MAX(order_purchase_timestamp)) - JULIANDAY(MIN(order_purchase_timestamp)) as days_active
    FROM customer_orders
    GROUP BY customer_unique_id
)
SELECT 
    customer_segment,
    COUNT(*) as customer_count,
    ROUND(AVG(total_orders), 2) as avg_orders,
    ROUND(AVG(days_active), 1) as avg_days_active,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as percentage
FROM customer_behavior
GROUP BY customer_segment
ORDER BY customer_count DESC;
"""

retention_results = pd.read_sql(retention_query, conn)
print("🎯 CUSTOMER RETENTION ANALYSIS")
print(retention_results)

🎯 CUSTOMER RETENTION ANALYSIS
  customer_segment  customer_count  avg_orders  avg_days_active  percentage
0         One-time           90557         1.0              0.0       97.00
1         Two-time            2573         2.0             82.0        2.76
2            Loyal             228         3.4            157.6        0.24


In [18]:
# 🔥 IMPRESSIVE QUERY #2: Monthly Cohort Retention Analysis
cohort_query = """
WITH customer_cohorts AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        DATE(o.order_purchase_timestamp, 'start of month') as order_month,
        MIN(DATE(o.order_purchase_timestamp, 'start of month')) OVER (PARTITION BY c.customer_unique_id) as cohort_month
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
),
cohort_data AS (
    SELECT 
        cohort_month,
        order_month,
        COUNT(DISTINCT customer_unique_id) as customers,
        (JULIANDAY(order_month) - JULIANDAY(cohort_month)) / 30 as period_number
    FROM customer_cohorts
    GROUP BY cohort_month, order_month
),
cohort_sizes AS (
    SELECT 
        cohort_month,
        COUNT(DISTINCT customer_unique_id) as cohort_size
    FROM customer_cohorts
    GROUP BY cohort_month
)
SELECT 
    cd.cohort_month,
    cs.cohort_size,
    cd.period_number,
    cd.customers,
    ROUND(100.0 * cd.customers / cs.cohort_size, 2) as retention_rate
FROM cohort_data cd
JOIN cohort_sizes cs ON cd.cohort_month = cs.cohort_month
WHERE cd.cohort_month >= '2017-01-01' AND cs.cohort_size >= 100
ORDER BY cd.cohort_month, cd.period_number;
"""

cohort_results = pd.read_sql(cohort_query, conn)
print("📊 COHORT RETENTION ANALYSIS")
print(cohort_results.head(20))

# Quick visualization
import matplotlib.pyplot as plt
pivot_cohort = cohort_results.pivot(index='cohort_month', columns='period_number', values='retention_rate')
print(f"\n📈 Cohort Retention Heatmap Preview:")
print(pivot_cohort.head())

📊 COHORT RETENTION ANALYSIS
   cohort_month  cohort_size  period_number  customers  retention_rate
0    2017-01-01          717       0.000000        717          100.00
1    2017-01-01          717       1.033333          2            0.28
2    2017-01-01          717       1.966667          2            0.28
3    2017-01-01          717       3.000000          1            0.14
4    2017-01-01          717       4.000000          3            0.42
5    2017-01-01          717       5.033333          1            0.14
6    2017-01-01          717       6.033333          3            0.42
7    2017-01-01          717       7.066667          1            0.14
8    2017-01-01          717       8.100000          1            0.14
9    2017-01-01          717      10.133333          3            0.42
10   2017-01-01          717      11.133333          1            0.14
11   2017-01-01          717      12.166667          5            0.70
12   2017-01-01          717      13.200000      

In [19]:
# 🔥 IMPRESSIVE QUERY #3: RFM Analysis (Recency, Frequency, Monetary)
rfm_query = """
WITH customer_rfm AS (
    SELECT 
        c.customer_unique_id,
        MAX(JULIANDAY('2018-10-17') - JULIANDAY(o.order_purchase_timestamp)) as recency_days,
        COUNT(DISTINCT o.order_id) as frequency,
        SUM(oi.price + oi.freight_value) as monetary_value
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
),
rfm_scores AS (
    SELECT 
        customer_unique_id,
        recency_days,
        frequency,
        monetary_value,
        NTILE(5) OVER (ORDER BY recency_days DESC) as recency_score,
        NTILE(5) OVER (ORDER BY frequency ASC) as frequency_score,
        NTILE(5) OVER (ORDER BY monetary_value ASC) as monetary_score
    FROM customer_rfm
),
rfm_segments AS (
    SELECT 
        *,
        recency_score + frequency_score + monetary_score as rfm_total,
        CASE 
            WHEN recency_score >= 4 AND frequency_score >= 4 THEN 'Champions'
            WHEN recency_score >= 4 AND frequency_score >= 2 THEN 'Loyal Customers'
            WHEN recency_score >= 3 AND frequency_score >= 3 THEN 'Potential Loyalists'
            WHEN recency_score >= 4 AND frequency_score = 1 THEN 'New Customers'
            WHEN recency_score = 1 AND frequency_score >= 4 THEN 'Cannot Lose Them'
            WHEN recency_score <= 2 AND frequency_score >= 2 THEN 'At Risk'
            ELSE 'Others'
        END as segment
    FROM rfm_scores
)
SELECT 
    segment,
    COUNT(*) as customer_count,
    ROUND(AVG(recency_days), 1) as avg_recency,
    ROUND(AVG(frequency), 2) as avg_frequency,
    ROUND(AVG(monetary_value), 2) as avg_monetary,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as percentage
FROM rfm_segments
GROUP BY segment
ORDER BY customer_count DESC;
"""

rfm_results = pd.read_sql(rfm_query, conn)
print("💎 RFM CUSTOMER SEGMENTATION")
print(rfm_results)

# Close connection
conn.close()
print("\n✅ DATABASE ANALYSIS COMPLETE!")

💎 RFM CUSTOMER SEGMENTATION
               segment  customer_count  avg_recency  avg_frequency  \
0              At Risk           22550        418.5           1.04   
1            Champions           14970        141.5           1.04   
2      Loyal Customers           14930        139.2           1.00   
3               Others           14885        358.4           1.00   
4  Potential Loyalists           11138        271.4           1.06   
5     Cannot Lose Them            7443        525.0           1.14   
6        New Customers            7442        139.1           1.00   

   avg_monetary  percentage  
0        163.60       24.15  
1        305.68       16.04  
2         90.61       15.99  
3         47.30       15.94  
4        224.43       11.93  
5        309.43        7.97  
6         39.63        7.97  

✅ DATABASE ANALYSIS COMPLETE!
